# Midterm 2 — Question 3

**Dataset:** Electricity 

**Regression:** `log(cost) ~ log(q) + log(pl) + log(pk) + log(pf)`

**Test:** H₀: β_log(pf) = 0.6 using a **non-robust** test at **1% significance level**

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

# Load the Electricity dataset (Nerlove 1963, from AER package)
import urllib.request
url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/AER/Electricity1955.csv"
urllib.request.urlretrieve(url, "Electricity.csv")

df = pd.read_csv("Electricity.csv")
print("Shape:", df.shape)
df.head()


Shape: (159, 9)


,rownames,cost,output,labor,laborshare,capital,capitalshare,fuel,fuelshare
0,1,0.082,2,2.09,0.3164,183.0,0.4521,17.9,0.2315
1,2,0.661,3,2.05,0.2073,174.0,0.6676,35.1,0.1251
2,3,0.990,4,2.05,0.2349,171.0,0.5799,35.1,0.1852
3,4,0.315,4,1.83,0.1152,166.0,0.7857,32.2,0.0990
4,5,0.197,5,2.12,0.2300,233.0,0.3841,28.6,0.3859


## Step 1: Create Log Variables

In [2]:
# Variable mapping: q=output, pl=labor, pk=capital, pf=fuel
df["log_cost"] = np.log(df["cost"])
df["log_q"]    = np.log(df["output"])
df["log_pl"]   = np.log(df["labor"])
df["log_pk"]   = np.log(df["capital"])
df["log_pf"]   = np.log(df["fuel"])

df[["log_cost", "log_q", "log_pl", "log_pk", "log_pf"]].describe()


,log_cost,log_q,log_pl,log_pk,log_pf
count,159.000000,159.000000,159.000000,159.000000,159.000000
mean,1.835660,6.670948,0.674363,5.154292,3.217722
std,1.441322,1.930940,0.121605,0.099075,0.348499
min,-2.501036,0.693147,0.371564,4.927254,2.332144
25%,0.923767,5.678428,0.565314,5.081404,3.111957
50%,1.971996,7.052721,0.712950,5.135798,3.292126
75%,2.833498,7.983049,0.781613,5.209486,3.490261
max,4.937505,9.861102,0.841567,5.451038,3.756538


## Step 2: Run OLS Regression (Non-Robust)

In [3]:
X = sm.add_constant(df[["log_q", "log_pl", "log_pk", "log_pf"]])
y = df["log_cost"]

model = sm.OLS(y, X).fit()  # non-robust (default)
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:               log_cost   R-squared:                       0.925
Model:                            OLS   Adj. R-squared:                  0.923
Method:                 Least Squares   F-statistic:                     472.1
Date:                Wed, 20 May 2026   Prob (F-statistic):           2.59e-85
Time:                        07:47:42   Log-Likelihood:                -77.725
No. Observations:                 159   AIC:                             165.5
Df Residuals:                     154   BIC:                             180.8
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -4.1581      1.760     -2.363      0.0

## Step 3: Hypothesis Test — H₀: β_log(pf) = 0.6

**Test statistic formula:**
$$t = \frac{\hat{\beta}_{log(pf)} - 0.6}{SE_{log(pf)}}$$

In [4]:
# Extract coefficient and standard error for log(pf)
beta_pf  = model.params["log_pf"]
se_pf    = model.bse["log_pf"]
h0_value = 0.6

# Test statistic
t_stat = (beta_pf - h0_value) / se_pf

# Critical value: two-tailed test, 1% significance, t-distribution
df_resid = model.df_resid
crit_val = stats.t.ppf(0.995, df=df_resid)  # 1 - alpha/2 = 1 - 0.01/2 = 0.995

print(f"Estimated beta_log(pf) : {beta_pf:.3f}")
print(f"Standard Error         : {se_pf:.3f}")
print(f"H0 value               : {h0_value}")
print(f"Degrees of freedom     : {int(df_resid)}")
print()
print(f"Test-statistic         : {t_stat:.3f}")
print(f"Critical value (+-)    : {crit_val:.3f}")
print()
if abs(t_stat) > crit_val:
    print("Conclusion: REJECT H0 — coefficient IS significantly different from 0.6 at 1%")
else:
    print("Conclusion: FAIL TO REJECT H0 — coefficient is NOT significantly different from 0.6 at 1%")


Estimated beta_log(pf) : 0.446
Standard Error         : 0.101
H0 value               : 0.6
Degrees of freedom     : 154

Test-statistic         : -1.524
Critical value (+-)    : 2.608

Conclusion: FAIL TO REJECT H0 — coefficient is NOT significantly different from 0.6 at 1%
